# RUKOPYS Qwen3-VL Inference and Submission

This notebook loads the LoRA adapter from the training notebook and writes `submission.csv`.

It uses a two-pass end-to-end pipeline:
- Pass 1: full page -> regions JSON (`bbox`, `type`, `text`).
- Pass 2: optional crop OCR for each predicted text region, using the same fine-tuned model, then replaces the weaker page-level text.

For best score use `CROP_OCR_MODE = "all_text"`. It is slower, but checkpointed per GPU so you can resume from partial CSVs.


In [ ]:
INSTALL_DEPS = True

if INSTALL_DEPS:
    import subprocess
    import sys

    commands = [
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--upgrade-strategy",
            "only-if-needed",
            "accelerate",
            "peft",
            "bitsandbytes",
            "qwen-vl-utils",
            "pandas==2.2.2",
            "pillow<12",
        ],
        [sys.executable, "-m", "pip", "install", "-q", "-U", "git+https://github.com/huggingface/transformers.git"],
    ]
    for cmd in commands:
        print("Running:", " ".join(cmd))
        subprocess.check_call(cmd)


In [ ]:
import gc
import json
import math
import os
import re
import shutil
import subprocess
from pathlib import Path

import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm

Image.MAX_IMAGE_PIXELS = None
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

BASE_MODEL_CANDIDATES = [
    "/kaggle/input/models/qwen-lm/qwen-3-vl/transformers/8b-instruct/1",
    "/kaggle/input/qwen3-vl-8b-instruct",
    "Qwen/Qwen3-VL-8B-Instruct",
]

LORA_CANDIDATES = [
    "/kaggle/working/qwen3vl_rukopys_curriculum/qwen3vl_rukopys_lora_final",
    "/kaggle/input/qwen3vl-rukopys-curriculum/qwen3vl_rukopys_lora_final",
    "/kaggle/input/qwen3vl-rukopys-lora/qwen3vl_rukopys_lora_final",
]

# Set this to your Kaggle dataset root for inference.
# Expected test structure:
#   DATASET_ROOT/test/images/*.jpg
#   DATASET_ROOT/test/metadata.jsonl
DATASET_ROOT = "/kaggle/input/YOUR_RUKOPYS_TEST_DATASET_SLUG"

VALIDATION_RECORDS_CANDIDATES = [
    "/kaggle/working/qwen3vl_rukopys_curriculum/gold_validation_records.jsonl",
    "/kaggle/input/qwen3vl-rukopys-curriculum/gold_validation_records.jsonl",
]

RUN_SPLIT = "test"  # choices: "test", "validation"
OUTPUT_CSV = "submission.csv"
TEST_MODE = False
CROP_OCR_MODE = "all_text"  # choices: "none", "smart", "all_text"
CROP_BATCH_SIZE = 2
CHECKPOINT_EVERY = 2

MAX_PIXELS_PAGE = 850_000
MAX_PIXELS_CROP = 262_144
MAX_NEW_TOKENS_PAGE = 4096
MAX_NEW_TOKENS_CROP = 192
CROP_PAD_RATIO = 0.04

PAGE_PROMPT = (
    "Extract every visible document region. Return only a compact JSON array. "
    "Each item must have keys bbox,type,text. bbox is [x1,y1,x2,y2] on a 0-1000 grid. "
    "type is one of handwritten,printed,formula,table,annotation,image,graph. "
    "Use empty text for image and graph. Preserve reading order."
)
CROP_PROMPTS = {
    "formula": "Read this formula region. Return only the formula text, LaTeX or plain Unicode. No explanation.",
    "table": "Read this table region. Return only pipe-separated table text. No explanation.",
    "annotation": "Read this short annotation. Return only the exact text. No explanation.",
    "default": "Transcribe this Ukrainian text region exactly. Return only the text. No explanation.",
}

VALID_TYPES = {"handwritten", "printed", "formula", "table", "annotation", "image", "graph"}
TEXT_TYPES = {"handwritten", "printed", "formula", "table", "annotation"}
IMAGE_EXTENSIONS = [".png", ".jpg", ".jpeg", ".webp", ".bmp"]


In [ ]:
def first_existing(paths):
    for item in paths:
        p = Path(item)
        if p.exists():
            return p
    return None


def find_model_id():
    for item in BASE_MODEL_CANDIDATES:
        if item.startswith("/") and Path(item).exists():
            return item
        if not item.startswith("/"):
            return item
    raise FileNotFoundError("No base model found.")


def find_lora_dir():
    for item in LORA_CANDIDATES:
        p = Path(item)
        if (p / "adapter_config.json").exists():
            return p
    for root, _, files in os.walk("/kaggle/input"):
        if "adapter_config.json" in files:
            return Path(root)
    raise FileNotFoundError("No LoRA adapter_config.json found. Add the training notebook output as Kaggle input.")


def get_dataset_root():
    root = Path(DATASET_ROOT)
    required_split = "train" if RUN_SPLIT == "validation" else "test"
    metadata_path = root / required_split / "metadata.jsonl"
    if not metadata_path.exists():
        raise FileNotFoundError(
            f"DATASET_ROOT is not configured correctly: {root}. "
            f"Expected {required_split}/metadata.jsonl under this path."
        )
    return root


def find_validation_records_path(lora_path):
    candidates = [Path(p) for p in VALIDATION_RECORDS_CANDIDATES]
    candidates.append(Path(lora_path).parent / "gold_validation_records.jsonl")
    candidates.append(Path(lora_path).parent.parent / "gold_validation_records.jsonl")
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError("No gold_validation_records.jsonl found. Run the training notebook or set RUN_SPLIT='test'.")


def read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def resolve_image_path(root, split, file_name):
    raw = Path(file_name)
    name = raw.name
    stem = raw.stem
    candidate_names = [name] + [stem + ext for ext in IMAGE_EXTENSIONS if stem + ext != name]
    candidates = [
        root / split / file_name,
        root / file_name,
    ]
    for candidate_name in candidate_names:
        candidates.extend([
            root / split / "images" / candidate_name,
            root / split / candidate_name,
        ])
    for p in candidates:
        if p.exists():
            return str(p)
    return str(root / split / "images" / candidate_names[0])


def load_prompt_config(lora_dir):
    global PAGE_PROMPT, CROP_PROMPTS, MAX_PIXELS_PAGE, MAX_PIXELS_CROP
    cfg_path = Path(lora_dir) / "rukopys_prompt_config.json"
    if not cfg_path.exists():
        return
    cfg = json.loads(cfg_path.read_text(encoding="utf-8"))
    PAGE_PROMPT = cfg.get("page_prompt", PAGE_PROMPT)
    CROP_PROMPTS.update(cfg.get("crop_prompts", {}))
    MAX_PIXELS_PAGE = int(cfg.get("max_pixels_page", MAX_PIXELS_PAGE))
    MAX_PIXELS_CROP = int(cfg.get("max_pixels_crop", MAX_PIXELS_CROP))


model_id = find_model_id()
lora_dir = find_lora_dir()
dataset_root = get_dataset_root()
load_prompt_config(lora_dir)

if RUN_SPLIT == "validation":
    validation_path = find_validation_records_path(lora_dir)
    test_records = read_jsonl(validation_path)
    IMAGE_SPLIT = "train"
    OUTPUT_CSV = "validation_pred.csv"
    print("Validation records:", validation_path)
else:
    test_records = read_jsonl(dataset_root / "test" / "metadata.jsonl")
    IMAGE_SPLIT = "test"
    OUTPUT_CSV = "submission.csv"
if TEST_MODE:
    test_records = test_records[:4]

print("Base model:", model_id)
print("LoRA:", lora_dir)
print("Dataset:", dataset_root)
print("Run split:", RUN_SPLIT)
print("Images:", len(test_records))


In [ ]:
def extract_json_array(text):
    text = (text or "").strip()
    if text.startswith("```json"):
        text = text[7:]
    elif text.startswith("```"):
        text = text[3:]
    if text.endswith("```"):
        text = text[:-3]
    text = text.strip()

    candidates = [text]
    m = re.search(r"\[.*\]", text, flags=re.DOTALL)
    if m:
        candidates.append(m.group(0))
    for cand in candidates:
        try:
            data = json.loads(cand)
            if isinstance(data, list):
                return data
            if isinstance(data, dict):
                for key in ("regions", "data", "items"):
                    if isinstance(data.get(key), list):
                        return data[key]
        except Exception:
            pass

    objects = []
    for m in re.finditer(r"\{[^{}]*\"bbox\"[^{}]*\}", text, flags=re.DOTALL):
        try:
            objects.append(json.loads(m.group(0)))
        except Exception:
            pass
    return objects


def normalize_type(value):
    value = str(value or "handwritten").strip().lower()
    return value if value in VALID_TYPES else "handwritten"


def bbox_to_pixels(box, width, height):
    if not isinstance(box, list) or len(box) != 4:
        return None
    try:
        vals = [float(v) for v in box]
    except Exception:
        return None
    max_val = max(vals)
    if max_val <= 1.5:
        x1, y1, x2, y2 = [vals[0] * width, vals[1] * height, vals[2] * width, vals[3] * height]
    elif max_val <= 1005:
        x1, y1, x2, y2 = [vals[0] / 1000 * width, vals[1] / 1000 * height, vals[2] / 1000 * width, vals[3] / 1000 * height]
    else:
        x1, y1, x2, y2 = vals
    x1, x2 = sorted((max(0, min(width, x1)), max(0, min(width, x2))))
    y1, y2 = sorted((max(0, min(height, y1)), max(0, min(height, y2))))
    if x2 - x1 < 3 or y2 - y1 < 3:
        return None
    return [int(round(x1)), int(round(y1)), int(round(x2)), int(round(y2))]


def iou(a, b):
    x1 = max(a[0], b[0])
    y1 = max(a[1], b[1])
    x2 = min(a[2], b[2])
    y2 = min(a[3], b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area_a = max(0, a[2] - a[0]) * max(0, a[3] - a[1])
    area_b = max(0, b[2] - b[0]) * max(0, b[3] - b[1])
    return inter / max(1, area_a + area_b - inter)


def clean_regions(raw_regions, image_path):
    with Image.open(image_path) as img:
        width, height = img.size
    cleaned = []
    for item in raw_regions:
        if not isinstance(item, dict):
            continue
        box = bbox_to_pixels(item.get("bbox"), width, height)
        if box is None:
            continue
        rtype = normalize_type(item.get("type"))
        text = "" if rtype in {"image", "graph"} else str(item.get("text") or "")
        cleaned.append({"bbox": box, "type": rtype, "text": text})

    cleaned.sort(key=lambda r: (r["bbox"][1], r["bbox"][0]))
    deduped = []
    for r in cleaned:
        duplicate = False
        for old in deduped:
            if iou(r["bbox"], old["bbox"]) > 0.88:
                duplicate = True
                if len(r.get("text", "")) > len(old.get("text", "")):
                    old.update(r)
                break
        if not duplicate:
            deduped.append(r)
    return deduped


def clean_crop_text(text):
    text = (text or "").strip()
    if text.startswith("```"):
        text = re.sub(r"^```[a-zA-Z]*", "", text).strip()
        text = re.sub(r"```$", "", text).strip()
    text = re.sub(r"^(text|transcription|answer)\s*:\s*", "", text, flags=re.I).strip()
    if len(text) >= 2 and text[0] == text[-1] and text[0] in {"'", '"'}:
        text = text[1:-1].strip()
    # Reject obvious non-answer JSON.
    if text.startswith("[") or text.startswith("{"):
        try:
            obj = json.loads(text)
            if isinstance(obj, dict) and "text" in obj:
                text = str(obj["text"])
            else:
                return ""
        except Exception:
            return ""
    return text[:500]


In [ ]:
from peft import PeftModel
from qwen_vl_utils import process_vision_info
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig


def configure_processor_for_generation(processor):
    if processor.tokenizer.pad_token_id is None:
        processor.tokenizer.pad_token = processor.tokenizer.eos_token
    processor.tokenizer.padding_side = "left"
    return processor


def load_worker_model(device):
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )
    base = AutoModelForImageTextToText.from_pretrained(
        model_id,
        device_map={"": device},
        quantization_config=quantization_config,
        dtype=torch.float16,
        trust_remote_code=True,
        attn_implementation="sdpa",
        low_cpu_mem_usage=True,
    )
    model = PeftModel.from_pretrained(base, str(lora_dir))
    model.eval()
    processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
    processor = configure_processor_for_generation(processor)
    if processor.tokenizer.pad_token_id is not None:
        model.generation_config.pad_token_id = processor.tokenizer.pad_token_id
    return model, processor


def generate_batch(model, processor, messages_batch, device, max_new_tokens):
    processor.tokenizer.padding_side = "left"
    texts = [
        processor.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
        for m in messages_batch
    ]
    image_inputs, video_inputs = process_vision_info(messages_batch)
    inputs = processor(
        text=texts,
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to(device)
    with torch.no_grad(), torch.amp.autocast("cuda", dtype=torch.float16):
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
        )
    trimmed = [o[len(i):] for i, o in zip(inputs.input_ids, out)]
    decoded = processor.batch_decode(trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)
    del inputs, out, trimmed
    torch.cuda.empty_cache()
    return decoded


def page_messages(image_path):
    return [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image_path, "max_pixels": MAX_PIXELS_PAGE},
                {"type": "text", "text": PAGE_PROMPT},
            ],
        }
    ]


def crop_image(image_path, bbox):
    with Image.open(image_path) as img:
        img = img.convert("RGB")
        w, h = img.size
        x1, y1, x2, y2 = bbox
        pad = int(round(max(x2 - x1, y2 - y1) * CROP_PAD_RATIO))
        x1 = max(0, x1 - pad)
        y1 = max(0, y1 - pad)
        x2 = min(w, x2 + pad)
        y2 = min(h, y2 + pad)
        return img.crop((x1, y1, x2, y2))


def should_crop_ocr(region):
    if CROP_OCR_MODE == "none":
        return False
    if region.get("type") not in TEXT_TYPES:
        return False
    if CROP_OCR_MODE == "all_text":
        return True
    text = str(region.get("text") or "")
    return (not text.strip()) or len(text) < 4 or len(text) > 160


def crop_messages(image_path, region):
    rtype = normalize_type(region.get("type"))
    prompt = CROP_PROMPTS.get(rtype, CROP_PROMPTS["default"])
    return [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": crop_image(image_path, region["bbox"]), "max_pixels": MAX_PIXELS_CROP},
                {"type": "text", "text": prompt},
            ],
        }
    ]


def infer_one_image(model, processor, image_path, device):
    raw = generate_batch(model, processor, [page_messages(image_path)], device, MAX_NEW_TOKENS_PAGE)[0]
    regions = clean_regions(extract_json_array(raw), image_path)

    crop_indices = [i for i, r in enumerate(regions) if should_crop_ocr(r)]
    for start in range(0, len(crop_indices), CROP_BATCH_SIZE):
        batch_indices = crop_indices[start:start + CROP_BATCH_SIZE]
        msgs = [crop_messages(image_path, regions[i]) for i in batch_indices]
        try:
            outs = generate_batch(model, processor, msgs, device, MAX_NEW_TOKENS_CROP)
        except Exception as e:
            print("Crop OCR batch failed:", e)
            continue
        for idx, out in zip(batch_indices, outs):
            text = clean_crop_text(out)
            if text:
                regions[idx]["text"] = text

    regions.sort(key=lambda r: (r["bbox"][1], r["bbox"][0]))
    return regions


In [ ]:
import multiprocessing as mp

PROGRESS_PRINT_EVERY = 1


def worker_process(gpu_id, records, output_csv):
    device = f"cuda:{gpu_id}"
    total_records = len(records)
    print(f"[GPU {gpu_id}] loading model for {total_records} images", flush=True)
    model, processor = load_worker_model(device)
    print(f"[GPU {gpu_id}] model loaded; starting inference", flush=True)

    done = set()
    results = []
    if Path(output_csv).exists():
        try:
            old = pd.read_csv(output_csv)
            old = old.drop_duplicates(subset=["image"], keep="last")
            done = set(old["image"].tolist())
            results = old.to_dict("records")
            print(f"[GPU {gpu_id}] resumed {len(done)}/{total_records} rows from {output_csv}", flush=True)
        except Exception as e:
            print(f"[GPU {gpu_id}] could not read checkpoint: {e}", flush=True)

    for rec in tqdm(records, desc=f"GPU {gpu_id}", position=gpu_id):
        image_name = Path(rec["file_name"]).name
        if image_name in done:
            continue
        image_path = resolve_image_path(dataset_root, IMAGE_SPLIT, rec["file_name"])
        try:
            regions = infer_one_image(model, processor, image_path, device)
        except Exception as e:
            print(f"[GPU {gpu_id}] failed {image_name}: {e}", flush=True)
            regions = []
            torch.cuda.empty_cache()
            gc.collect()
        results.append({"image": image_name, "regions": json.dumps(regions, ensure_ascii=False)})
        done.add(image_name)

        done_count = len(done)
        if done_count % PROGRESS_PRINT_EVERY == 0 or done_count == total_records:
            pct = done_count / max(1, total_records)
            print(
                f"[GPU {gpu_id}] progress {done_count}/{total_records} ({pct:.1%}) "
                f"last={image_name} regions={len(regions)}",
                flush=True,
            )

        if len(results) % CHECKPOINT_EVERY == 0:
            tmp = output_csv + ".tmp"
            pd.DataFrame(results).drop_duplicates(subset=["image"], keep="last").to_csv(tmp, index=False)
            os.replace(tmp, output_csv)
            print(f"[GPU {gpu_id}] checkpoint saved {output_csv} rows={len(results)}", flush=True)

    tmp = output_csv + ".tmp"
    pd.DataFrame(results).drop_duplicates(subset=["image"], keep="last").to_csv(tmp, index=False)
    os.replace(tmp, output_csv)
    print(f"[GPU {gpu_id}] done {len(done)}/{total_records}; saved {output_csv}", flush=True)


def detect_num_gpus():
    try:
        out = subprocess.check_output(["nvidia-smi", "-L"]).decode("utf-8").strip()
        return max(1, len([x for x in out.splitlines() if x.strip()]))
    except Exception:
        return max(1, torch.cuda.device_count())


mp.set_start_method("fork", force=True)
num_gpus = detect_num_gpus()
print("GPUs:", num_gpus, flush=True)

chunks = []
chunk_size = math.ceil(len(test_records) / num_gpus)
for i in range(num_gpus):
    chunks.append(test_records[i * chunk_size:(i + 1) * chunk_size])

processes = []
partials = []
for gpu_id, chunk in enumerate(chunks):
    if not chunk:
        continue
    output_csv = f"partial_results_gpu{gpu_id}.csv"
    partials.append(output_csv)
    p = mp.Process(target=worker_process, args=(gpu_id, chunk, output_csv))
    p.start()
    processes.append(p)

for p in processes:
    p.join()

frames = []
for path in partials:
    if Path(path).exists():
        frame = pd.read_csv(path)
        print(f"Partial {path}: rows={len(frame)}", flush=True)
        frames.append(frame)
if not frames:
    raise RuntimeError("No partial outputs were created.")

final = pd.concat(frames, ignore_index=True).drop_duplicates(subset=["image"], keep="last")
order = [Path(r["file_name"]).name for r in test_records]
final = final.set_index("image").reindex(order).reset_index()
final["regions"] = final["regions"].fillna("[]")
final.to_csv(OUTPUT_CSV, index=False)
print("Wrote", OUTPUT_CSV, "rows=", len(final), flush=True)
final.head()


In [ ]:
# Submission sanity check.
df = pd.read_csv(OUTPUT_CSV)
assert list(df.columns) == ["image", "regions"], df.columns
assert len(df) == len(test_records), (len(df), len(test_records))
bad = []
for row in df.itertuples(index=False):
    try:
        parsed = json.loads(row.regions)
        assert isinstance(parsed, list)
        for item in parsed:
            assert "bbox" in item and "type" in item and "text" in item
    except Exception as e:
        bad.append((row.image, str(e)))
        if len(bad) >= 5:
            break
print("Bad rows:", bad[:5])
print("Ready:", OUTPUT_CSV)

if RUN_SPLIT == "validation":
    solution_rows = []
    for rec in test_records:
        solution_rows.append({
            "image": Path(rec["file_name"]).name,
            "regions": json.dumps(rec.get("regions") or [], ensure_ascii=False),
        })
    solution_df = pd.DataFrame(solution_rows)
    try:
        from kaggle_metric import score_detailed

        breakdown = score_detailed(solution_df, df, "image")
        print("Local validation breakdown:", breakdown)
    except Exception as e:
        print("Skipped local metric. Add kaggle_metric.py from the official metric notebook to score locally.")
        print("Reason:", e)
